## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simple, brute-force type of RAG..

### Sidenote: Business applications of this week's projects

RAG is perhaps the most immediately applicable technique of anything that we cover in the course! In fact, there are commercial products that do precisely what we build this week: nuanced querying across large databases of information, such as company contracts or product specs. RAG gives you a quick-to-market, low cost mechanism for adapting an LLM to your business area.

In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

### Nhập các thư viện cần thiết:
- os: Cung cấp các chức năng để làm việc với hệ điều hành, như đọc biến môi trường.

- glob: Dùng để tìm kiếm file theo mẫu (pattern).

- dotenv.load_dotenv(): Tải các biến môi trường từ file .env để tránh ghi trực tiếp API key vào code.

- gradio: Thư viện giúp tạo giao diện người dùng AI một cách dễ dàng.

- OpenAI: Thư viện của OpenAI để gọi API (GPT, DALL·E, v.v.).

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"

### Định nghĩa mô hình AI
Xác định mô hình AI sẽ sử dụng, ở đây là "gpt-4o-mini", một phiên bản tiết kiệm chi phí hơn của GPT-4o.

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
openai = OpenAI()

### Tải biến môi trường và khởi tạo OpenAI
- load_dotenv(override=True): Tải biến môi trường từ tệp .env, ghi đè các biến có sẵn.

- os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env'):

- Lấy khóa API OpenAI từ biến môi trường, nếu không có thì dùng giá trị mặc định 'your-key-if-not-using-env'.

- openai = OpenAI(): Khởi tạo một đối tượng OpenAI để giao tiếp với mô hình AI.

In [ ]:
# With massive thanks to student Dr John S. for fixing a bug in the below for Windows users!

context = {}

employees = glob.glob("knowledge-base/employees/*")

for employee in employees:
    name = employee.split(' ')[-1][:-3]
    doc = ""
    with open(employee, "r", encoding="utf-8") as f:
        doc = f.read()
    context[name]=doc

### Tạo context từ tệp nhân viên
- context = {}: Khởi tạo một dictionary để lưu trữ dữ liệu.

- employees = glob.glob("knowledge-base/employees/*"):

   - Lấy danh sách tất cả các tệp trong thư mục knowledge-base/employees/.

- Vòng lặp for employee in employees:

   - Lấy tên nhân viên từ đường dẫn tệp (bằng cách cắt chuỗi).
 
   - Mở và đọc nội dung của tệp.

   - Lưu nội dung vào dictionary context với key là tên nhân viên.

In [ ]:
context["Lancaster"]

### Truy vấn dữ liệu nhân viên
- Truy xuất dữ liệu từ dictionary context với key "Lancaster".

- Điều này có nghĩa là đang lấy thông tin về nhân viên có tên "Lancaster".

In [ ]:
products = glob.glob("knowledge-base/products/*")

for product in products:
    name = product.split(os.sep)[-1][:-3]
    doc = ""
    with open(product, "r", encoding="utf-8") as f:
        doc = f.read()
    context[name]=doc

### Tạo context từ tệp sản phẩm
- products = glob.glob("knowledge-base/products/*"):

  - Lấy danh sách tất cả các tệp trong thư mục knowledge-base/products/.

- Vòng lặp for product in products:

  - Lấy tên sản phẩm từ đường dẫn tệp.

  - Đọc nội dung của tệp.

  - Lưu nội dung vào dictionary context với key là tên sản phẩm.


In [ ]:
context.keys()

### Lấy danh sách các key trong context
- context.keys(): Trả về danh sách tất cả các key trong dictionary context.

- Key này có thể là tên nhân viên hoặc sản phẩm đã lưu từ trước.

In [ ]:
system_message = "You are an expert in answering accurate questions about Insurellm, the Insurance Tech company. Give brief, accurate answers. If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context."

In [ ]:
def get_relevant_context(message):
    relevant_context = []
    for context_title, context_details in context.items():
        if context_title.lower() in message.lower():
            relevant_context.append(context_details)
    return relevant_context          

###  Định nghĩa thông điệp hệ thống
- Xác định thông điệp hệ thống (system_message) cho AI.

- AI được hướng dẫn trả lời câu hỏi liên quan đến công ty Insurellm.

- Yêu cầu trả lời ngắn gọn, chính xác, và không bịa đặt nếu không có dữ liệu phù hợp.

In [ ]:
get_relevant_context("Who is lancaster?")

### Hàm tìm kiếm ngữ cảnh liên quan
- Mục đích: Tìm kiếm các dữ liệu trong context có liên quan đến câu hỏi của người dùng.

- Cách hoạt động:

  - Duyệt qua toàn bộ context.

  - Nếu context_title (tên nhân viên hoặc sản phẩm) xuất hiện trong message, nội dung tương ứng (context_details) sẽ được thêm vào relevant_context.

  - Trả về danh sách relevant_context.

In [ ]:
get_relevant_context("Who is Avery and what is carllm?")

### Truy vấn ngữ cảnh từ câu hỏi
- Gọi hàm get_relevant_context() với câu hỏi cụ thể.

- Kiểm tra xem "Lancaster" hoặc "Avery" có xuất hiện trong danh sách dữ liệu (context) hay không.

- Nếu có, trả về nội dung liên quan.

In [ ]:
def add_context(message):
    relevant_context = get_relevant_context(message)
    if relevant_context:
        message += "\n\nThe following additional context might be relevant in answering this question:\n\n"
        for relevant in relevant_context:
            message += relevant + "\n\n"
    return message

### Thêm ngữ cảnh vào tin nhắn người dùng
- Mục đích: Cải thiện câu hỏi của người dùng bằng cách tự động thêm ngữ cảnh liên quan.

- Cách hoạt động:

  1. Gọi get_relevant_context(message) để tìm các dữ liệu liên quan.

  2. Nếu có ngữ cảnh liên quan, thêm một đoạn ghi chú "The following additional context might be relevant..." vào tin nhắn.

  3. Nối các dữ liệu tìm thấy vào cuối tin nhắn.

  4. Trả về tin nhắn đã mở rộng.

In [ ]:
print(add_context("Who is Alex Lancaster?"))

### Kiểm tra hàm add_context()
- Hàm add_context() được gọi với câu hỏi "Who is Alex Lancaster?".

- Nếu "Alex Lancaster" có trong context, chương trình sẽ trả về câu hỏi kèm theo thông tin bổ sung.

- In kết quả ra màn hình để kiểm tra.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history
    message = add_context(message)
    messages.append({"role": "user", "content": message})

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

### Hàm trò chuyện (chat())
- Mục đích: Tạo một chatbot sử dụng OpenAI API.

- Cách hoạt động:

  1. Khởi tạo tin nhắn:

     - messages = [{"role": "system", "content": system_message}] + history:

      - Thêm thông điệp hệ thống vào lịch sử trò chuyện (history).

  2. Thêm ngữ cảnh vào câu hỏi:

     - message = add_context(message):

      - Nếu có dữ liệu liên quan, chương trình sẽ bổ sung vào câu hỏi.

  3. Gửi câu hỏi tới mô hình AI:

     - messages.append({"role": "user", "content": message})

      - Sử dụng OpenAI API để tạo phản hồi (openai.chat.completions.create(...)).

  4. Xử lý phản hồi:

     - Dữ liệu được gửi về theo luồng (stream=True).

      - Gộp từng phần phản hồi lại và trả về kết quả cuối cùng (yield response).

## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch()

### Tạo giao diện Gradio
- gr.ChatInterface(chat, type="messages"):

   - Tạo một giao diện chatbot sử dụng Gradio.

   - Liên kết với hàm chat() để xử lý hội thoại.

- .launch():

    Khởi chạy ứng dụng chatbot.

